# Construção e auditoria da base paralela de custos
Somente referências de custos. Nenhuma regressão, imputação ou atribuição municipal. Estratégia e critérios em ESTRATEGIA.md e README.md.

## 1. Reconstruir a partir das fontes preservadas
Entradas copiadas e verificadas por SHA256; valores nominais e células originais preservados.

In [1]:
import json, unittest
from pathlib import Path
import construir as c
c.construir()
resumo = json.loads(Path("resumo.json").read_text())
resumo

{"series_estritas_auditoria_anterior": 87, "series": 188, "agregados_ha": 1880, "series_estritas": 85, "agregados_estritos": 850, "componentes": 7027, "componentes_elegiveis": 2938, "blocos_componentes": 1128, "blocos_aprovados": 1124, "identidades_aprovadas": 752, "identidades": 752, "celulas_fontes": 45360, "cpi_base_media2024": 313.6888333333333, "usd2024_preenchidos": 1790, "usd2024_estritos": 850, "imea_linhas": 585, "imea_candidatos_descritivos": 195, "sem_regressoes": true}


{'series_estritas_auditoria_anterior': 87,
 'series': 188,
 'agregados_ha': 1880,
 'series_estritas': 85,
 'agregados_estritos': 850,
 'componentes': 7027,
 'componentes_elegiveis': 2938,
 'blocos_componentes': 1128,
 'blocos_aprovados': 1124,
 'identidades_aprovadas': 752,
 'identidades': 752,
 'celulas_fontes': 45360,
 'cpi_base_media2024': 313.6888333333333,
 'usd2024_preenchidos': 1790,
 'usd2024_estritos': 850,
 'imea_linhas': 585,
 'imea_candidatos_descritivos': 195,
 'sem_regressoes': True}

## 2. Entender o suporte efetivo
Dez conceitos por planilha não são dez unidades independentes. Soja e sistemas de milho permanecem distintos.

In [2]:
c.ler(Path("cobertura.csv"))

[{'0_cultura': 'milho_1a',
  '0_conjunto': 'todas',
  '0_series': '17',
  '0_pracas': '1',
  '0_safra_inicio_min': '1997',
  '0_safra_fim_max': '2014',
  '0_tecnologias': 'não declarada'},
 {'0_cultura': 'milho_1a',
  '0_conjunto': 'estritas',
  '0_series': '9',
  '0_pracas': '1',
  '0_safra_inicio_min': '2001',
  '0_safra_fim_max': '2010',
  '0_tecnologias': 'não declarada'},
 {'0_cultura': 'milho_1a',
  '0_conjunto': 'quarentena',
  '0_series': '8',
  '0_pracas': '1',
  '0_safra_inicio_min': '1997',
  '0_safra_fim_max': '2014',
  '0_tecnologias': 'não declarada'},
 {'0_cultura': 'milho_2a',
  '0_conjunto': 'todas',
  '0_series': '72',
  '0_pracas': '4',
  '0_safra_inicio_min': '1997',
  '0_safra_fim_max': '2025',
  '0_tecnologias': 'OGM;não declarada'},
 {'0_cultura': 'milho_2a',
  '0_conjunto': 'estritas',
  '0_series': '44',
  '0_pracas': '4',
  '0_safra_inicio_min': '2001',
  '0_safra_fim_max': '2024',
  '0_tecnologias': 'OGM;não declarada'},
 {'0_cultura': 'milho_2a',
  '0_conjun

## 3. Reconciliar componentes e separar quarentena
Tolerância contábil 0,05 BRL/ha; teste ausente ou hierarquia ambígua não é aprovação. A aprovação anterior de87 séries é preservada como flag; o critério adicional resulta em85.

In [3]:
[r for r in c.ler(Path("auditoria_componentes.csv")) if r["4_soma_aprovada"] != "True"]

[{'0_id_serie': '6a58c6f0f5837c30',
  '0_bloco_conab': 'A',
  '0_n_componentes': '25',
  '2_soma_componentes': '937.66',
  '2_agregado_original': '976.78',
  '2_residuo': '-39.120000000000005',
  '4_verificacao_completa': 'True',
  '4_soma_aprovada': 'False',
  '4_tolerancia_brl_ha': '0.05'},
 {'0_id_serie': 'acc833dcae80ab00',
  '0_bloco_conab': 'A',
  '0_n_componentes': '25',
  '2_soma_componentes': '',
  '2_agregado_original': '1162.26',
  '2_residuo': '',
  '4_verificacao_completa': 'False',
  '4_soma_aprovada': 'False',
  '4_tolerancia_brl_ha': '0.05'},
 {'0_id_serie': 'f121da66bd98194b',
  '0_bloco_conab': 'A',
  '0_n_componentes': '25',
  '2_soma_componentes': '',
  '2_agregado_original': '876.14',
  '2_residuo': '',
  '4_verificacao_completa': 'False',
  '4_soma_aprovada': 'False',
  '4_tolerancia_brl_ha': '0.05'},
 {'0_id_serie': 'aa1f472c7d1d1b87',
  '0_bloco_conab': 'A',
  '0_n_componentes': '25',
  '2_soma_componentes': '',
  '2_agregado_original': '1019.16',
  '2_residuo':

## 4. Conferir moeda e dados ausentes
Valores em USD2024 somente quando data-base, câmbio e CPI forem válidos. Preservar nominal; não usar custos IPCA2024 como se fossem nominais. Ver status e proveniência da conversão.

In [4]:
c.ler(Path("qualidade_variaveis.csv"))

[{'0_variavel': '2_custo_brl_nominal_ha',
  '0_observados': '1880',
  '0_ausentes': '0',
  '0_zeros': '0',
  '2_minimo': '4.67',
  '2_maximo': '8053.370000000001'},
 {'0_variavel': '2_produtividade_modal_kg_ha',
  '0_observados': '1880',
  '0_ausentes': '0',
  '0_zeros': '0',
  '2_minimo': '2800.0',
  '2_maximo': '7200.0'},
 {'0_variavel': '2_cambio_brl_usd',
  '0_observados': '1880',
  '0_ausentes': '0',
  '0_zeros': '0',
  '2_minimo': '1.0683',
  '2_maximo': '5.7468'},
 {'0_variavel': '2_cpi_u_mes',
  '0_observados': '1790',
  '0_ausentes': '90',
  '0_zeros': '0',
  '2_minimo': '160.1',
  '2_maximo': '312.332'},
 {'0_variavel': '2_custo_usd2024_ha',
  '0_observados': '1790',
  '0_ausentes': '90',
  '0_zeros': '0',
  '2_minimo': '3.1739817514042525',
  '2_maximo': '1872.5505198472765'}]

## 5. IMEA: suporte e limites de comparabilidade
Valores declarados em R$/ha;sem conversão por falta de data nominal ou CPI2026. O subconjunto descritivo não é amostra estrita comparável à CONAB.

In [5]:
json.loads(Path("resumo_imea.json").read_text())

{'registros_planilha_coluna': 195,
 'abas': 28,
 'celulas': 585,
 'numericas': 525,
 'ausentes': 60,
 'candidatos_descritivos': 195,
 'registros_safra_ambigua': 2,
 'comparaveis_estritos': 0,
 'usd2024': 0,
 'nota': '585 células são195 unidades planilha/coluna; não585 amostras independentes. Quarentena integral para comparabilidade; subset descritivo não aprovação plena.'}

## 6. Testar identidades e preservação
Verificações incluem arquivos base3 v1–v4 publicados; não há alteração desses arquivos.

In [6]:
import test_base_custos
suite = unittest.defaultTestLoader.loadTestsFromModule(test_base_custos)
resultado = unittest.TextTestRunner(verbosity=2).run(suite)
assert resultado.wasSuccessful()

test_chaves_e_particao (test_base_custos.TestCustos.test_chaves_e_particao) ... 

ok


test_componentes (test_base_custos.TestCustos.test_componentes) ... 

ok


test_contabilidade_e_moeda (test_base_custos.TestCustos.test_contabilidade_e_moeda) ... 

...

test_preservacao_fontes_e_base3 (test_base_custos.TestCustos.test_preservacao_fontes_e_base3) ... 

ok


test_valores_originais (test_base_custos.TestCustos.test_valores_originais) ... 

ok


----------------------------------------------------------------------
Ran 7 tests in 0.927s

OK


## 7. Reproduzir duas vezes em diretórios isolados
O construtor não depende de rede ou do complemento externo após cópia das entradas.

In [7]:
from verificar_reproducao import verificar
repro = verificar()
[{"hashes_iguais": r["hashes_iguais"], "numero_csv": r["numero_csv"]} for r in repro["execucoes_isoladas"]]

[{'hashes_iguais': True, 'numero_csv': 16},
 {'hashes_iguais': True, 'numero_csv': 16}]

## 8. O que pode ser incorporado no futuro
Referência modal não é gasto municipal observado. Harmonização CONAB/IMEA depende dos componentes e da data. Crédito não substitui custo e frete/juros não podem ser somados novamente quando já incluídos.

In [8]:
c.ler(Path("candidatos_incorporacao.csv"))

[{'variavel': 'CONAB_H_operacional_nominal',
  'status': 'apto_referencia',
  'uso': '85 referências de cultura/safra/praça/tecnologia',
  'pendencia': 'Não transferível automaticamente ao município'},
 {'variavel': 'CONAB_J_total_nominal',
  'status': 'apto_referencia',
  'uso': 'Retorno incluindo renda dos fatores; distinto de H',
  'pendencia': 'Não somar novamente juros/frete já incluídos'},
 {'variavel': 'CONAB_componentes',
  'status': 'apto_referencia',
  'uso': '2938 componentes elegíveis; preservar rubricas',
  'pendencia': 'Harmonização entre fontes não validada'},
 {'variavel': 'USD2024_ha',
  'status': 'apto_referencia',
  'uso': '850 valores estritos em USD2024/ha',
  'pendencia': 'IMEA sem data nominal suficiente ou CPI2026;nenhum custo municipal'},
 {'variavel': 'IMEA_COE_COT_CT',
  'status': 'apenas_descricao_sensibilidade_condicional',
  'uso': '195 valores de65 registros com safra-alvo/valor/rótulo coerentes',
  'pendencia': 'Data nominal consolidada,componentes e com